## Tardis Model

### 1. Package Importation

In [22]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import joblib
from datetime import datetime
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.linear_model import LinearRegression
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

sns.set_theme(style="whitegrid")

In [23]:
df = pd.read_csv("cleaned_dataset.csv")
cleaned_df = df

In [24]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 9280 entries, 0 to 9279
Data columns (total 26 columns):
 #   Column                                                                                       Non-Null Count  Dtype  
---  ------                                                                                       --------------  -----  
 0   Date                                                                                         9280 non-null   object 
 1   Service                                                                                      9280 non-null   object 
 2   Gare de départ                                                                               9280 non-null   object 
 3   Gare d'arrivée                                                                               9280 non-null   object 
 4   Durée moyenne du trajet                                                                      9280 non-null   float64
 5   Nombre de circulations prévues    

In [25]:
df.isnull().sum()

Date                                                                                           0
Service                                                                                        0
Gare de départ                                                                                 0
Gare d'arrivée                                                                                 0
Durée moyenne du trajet                                                                        0
Nombre de circulations prévues                                                                 0
Nombre de trains annulés                                                                       0
Commentaire annulations                                                                        0
Nombre de trains en retard au départ                                                           0
Retard moyen des trains en retard au départ                                                    0
Retard moyen de tous les train

### 1.1 Analysis on a global basis

In [26]:
#df = df.select_dtypes(exclude=['object'])
df = pd.get_dummies(df, columns=["Gare de départ", "Gare d'arrivée", "Service","Commentaire annulations","Commentaire retards à l'arrivée","Commentaire retards au départ"], drop_first=True)

df["Date"] = pd.to_datetime(df["Date"], format="%Y-%m")

df["Annee"] = df["Date"].dt.year
df["Mois"] = df["Date"].dt.month

df = df.drop("Date", axis=1)


#df = pd.get_dummies(df, columns=["Gare de départ", "Gare d'arrivée", "Service"], drop_first=True)

X = df.drop("Retard moyen de tous les trains à l'arrivée",axis=1)
y = df["Retard moyen de tous les trains à l'arrivée"]

X_train,X_test,y_train,y_test = train_test_split(X,y,test_size=0.2,random_state=42)

In [27]:
pipe = make_pipeline(StandardScaler(),LinearRegression())
pipe.fit(X_train, y_train) 


predictions = pipe.predict(X_test)   


print("MAE:", mean_absolute_error(y_test, predictions))
print("RMSE:", np.sqrt(mean_squared_error(y_test, predictions)))
print("R² Score:", r2_score(y_test, predictions))    

MAE: 184539626248.36093
RMSE: 1696674290884.6045
R² Score: -2.9988011502514257e+23


In [28]:
joblib.dump( 
    { 
        "model": pipe, 
        "features": list(X.columns),
        "X_columns": list(X.columns),
    },
    "model.pkl",
)

['model.pkl']

In [29]:
df_test = pd.DataFrame({
    "Gare de départ": ["LE MANS"],
    "Gare d'arrivée": ["PARIS MONTPARNASSE"],
    "Service": ["National"],
    "Date": ["2027-08"],
    "Durée moyenne du trajet": [None],
    "Nombre de circulations prévues": [None],
    "Nombre de trains annulés": [None],
    "Commentaire annulations": [None],
    "Nombre de trains en retard au départ": [None],
    "Retard moyen des trains en retard au départ": [None],
    "Retard moyen de tous les trains au départ": [None],
    "Commentaire retards au départ": [None],
    "Nombre de trains en retard à l'arrivée": [None],
    "Retard moyen des trains en retard à l'arrivée": [None],
    "Commentaire retards à l'arrivée": [None],
    "Nombre trains en retard > 15min": [None],
    "Retard moyen trains en retard > 15 (si liaison concurrencée par vol)": [None],
    "Nombre trains en retard > 30min": [None],
    "Nombre trains en retard > 60min": [None],
    "Prct retard pour causes externes": [None],
    "Prct retard pour cause infrastructure": [None],
    "Prct retard pour cause gestion trafic": [None],
    "Prct retard pour cause matériel roulant": [None],
    "Prct retard pour cause gestion en gare et réutilisation de matériel": [None],
    "Prct retard pour cause prise en compte voyageurs (affluence, gestions PSH, correspondances)": [None]
})


def search_similate(cdf,df):

    GD = df["Gare de départ"].iloc[0]
    GA = df["Gare d'arrivée"].iloc[0]
    S = df["Service"].iloc[0]
    Date = df["Date"].iloc[0]
    
    target_date = datetime.strptime(Date, "%Y-%m")
    year = target_date.year
    month = target_date.month

    df["Annee"] = year
    df["Mois"] = month
    
    cdf["Date"] = pd.to_datetime(cdf["Date"], format="%Y-%m")
    
    cdf["Annee"] = cdf["Date"].dt.year
    cdf["Mois"] = cdf["Date"].dt.month
    
    filters = [
        (cdf["Gare d'arrivée"] == GA),
        (cdf["Mois"] == month),
        (cdf["Gare de départ"] == GD),
        (cdf["Annee"] == year),
        (cdf["Service"] == S)
    ]
    
    for i in range(len(filters), 0, -1):
        mask = filters[:i][0]
        for filter in filters[:i][1:]:
            mask = mask & filter        
        result = cdf[mask]  
        print(result)      
        if not result.empty:
            for col in df.columns:
                if pd.isna(df[col].iloc[0]) and col in result.columns:
                    df.at[0, col] = result.iloc[0][col]
            return True
    
    print("Aucun match trouvé")
    return False


search_similate(
    cleaned_df,
    df_test
)



df_test = pd.get_dummies(
    df_test,
    columns=[
        "Gare de départ",
        "Gare d'arrivée",
        "Service",
        "Commentaire annulations",
        "Commentaire retards à l'arrivée",
        "Commentaire retards au départ"
    ],
    drop_first=True
)

df_test = df_test.drop("Date", axis=1)

df_test = df_test.reindex(columns=X.columns, fill_value=0)

predictions = pipe.predict(df_test)

print( "predictions : ",predictions[0])


Empty DataFrame
Columns: [Date, Service, Gare de départ, Gare d'arrivée, Durée moyenne du trajet, Nombre de circulations prévues, Nombre de trains annulés, Commentaire annulations, Nombre de trains en retard au départ, Retard moyen des trains en retard au départ, Retard moyen de tous les trains au départ, Commentaire retards au départ, Nombre de trains en retard à l'arrivée, Retard moyen des trains en retard à l'arrivée, Retard moyen de tous les trains à l'arrivée, Commentaire retards à l'arrivée, Nombre trains en retard > 15min, Retard moyen trains en retard > 15 (si liaison concurrencée par vol), Nombre trains en retard > 30min, Nombre trains en retard > 60min, Prct retard pour causes externes, Prct retard pour cause infrastructure, Prct retard pour cause gestion trafic, Prct retard pour cause matériel roulant, Prct retard pour cause gestion en gare et réutilisation de matériel, Prct retard pour cause prise en compte voyageurs (affluence, gestions PSH, correspondances), Annee, Mois]
